In [ ]:
# 1. 安装依赖 + 挂载云盘
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')
print("云盘已挂载")

In [ ]:
# 2. 解压数据
import os, glob
!cp /content/drive/MyDrive/garbage_dataset.zip /content/
!unzip -qo /content/garbage_dataset.zip -d /content/dataset/
print(f"数据已解压: {len(glob.glob('/content/dataset/garbage_classification/*/*'))} 个文件")

In [ ]:
# 3. 生成合成数据（如果已有可跳过）
import os
if os.path.exists('/content/dataset/yolo_v3/dataset.yaml'):
    print("数据已存在，跳过生成")
else:
    print("需要先生成数据...")  # will generate below

In [ ]:
# 4. GPU训练
!pip install ultralytics -q
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
print("开始训练...")
results = model.train(
    data='/content/dataset/yolo_v3/dataset.yaml',
    epochs=50, batch=32, imgsz=640,
    device=0, workers=4,
    name='garbage-multi',
    patience=10, lr0=0.005, seed=42,
)
print(f"训练完成! 最佳模型: {results.save_dir}/weights/best.pt")

In [ ]:
# 5. 测试 + 导出到云盘
!pip install ultralytics -q
from ultralytics import YOLO
import glob, os

model = YOLO('runs/detect/garbage-multi/weights/best.pt')
test_imgs = glob.glob('/content/dataset/yolo_v3/val/images/comp_*')[:10]
for p in test_imgs:
    r = model(p, conf=0.25)[0]
    if r.boxes and len(r.boxes) > 0:
        dets = [(model.names[int(b.cls)], f"{float(b.conf):.2f}") for b in r.boxes]
        print(f"{os.path.basename(p)}: {len(r.boxes)}个 -> {dets}")

!cp runs/detect/garbage-multi/weights/best.pt /content/drive/MyDrive/best_multi.pt
print()
print("模型已导出到: /content/drive/MyDrive/best_multi.pt")
print("从云盘下载 best_multi.pt 放到项目 models/best.pt 即可")